In [ ]:
import glob, importlib, os, shutil, subprocess, sys
from pathlib import Path
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')
for pat in ('/usr/local/lib/python*/dist-packages/*cutlass*.pth', '/usr/local/lib/python*/site-packages/*cutlass*.pth'):
    for pth in glob.glob(pat):
        try: os.unlink(pth)
        except OSError: pass
tmp = Path('/tmp/wheelhouse'); tmp.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower(): continue
    name = w.name.replace('cu128', '+cu128') if ('cu128' in w.name and '+' not in w.name) else w.name
    t = tmp / name
    if not t.exists(): os.symlink(w, t)
wheels = sorted(str(w) for w in tmp.glob('*.whl'))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels], check=True)
importlib.invalidate_caches()
print('installed', len(wheels), 'wheels')

In [ ]:
import asyncio, json, logging, time, traceback
from adk_eval_core.tracing import SessionTrace
from adk_submission import ModelRegistry, discover_declared_models
from google.adk.models.lite_llm import LiteLlm
from swegemma.config import EvalConfig
from swegemma.deduplication import resolve_task_snapshot_paths
from swegemma.evaluate import Evaluator
from swegemma.harness.verification import verify_task
from swegemma.models import load_tasks

class _DropEnsurepipNoise(logging.Filter):
    def filter(self, record): return 'ensurepip unavailable' not in record.getMessage()
logging.getLogger('swegemma.sandbox.subprocess').addFilter(_DropEnsurepipNoise())

POOL = Path(next(Path('/kaggle/input').rglob('tasks.jsonl'))).parent
ARM = next(p.parent for p in Path('/kaggle/input').rglob('agent.yaml') if 'swe_scaffold_v2_b1' in p.read_text(encoding='utf-8'))
print('pool:', POOL, '| arm:', ARM, '| wheels:', len(list((POOL / 'wheels').glob('*.whl'))), '| snapshots:', len(list((POOL / 'snapshots').glob('*.tgz'))))
WORK = Path('/kaggle/working'); (WORK / 'gold').mkdir(exist_ok=True)
# In subprocess mode swegemma does NOT inject the wheel set (install_test_dependencies returns early); the sandbox venv only sees the host's site-packages through a
# _host_env.pth built from sys.path. So unpack the shared wheels into a side directory and APPEND it to sys.path (host packages keep priority).
import subprocess, sys
exec("\"\"\"Unpack the shared wheel set into a side directory for the subprocess sandbox (used by the gold check and by the rollout notebooks).\n\nFacts established from the swegemma 0.2.11 sources and the pilots (2026-10-13):\n  * in subprocess mode swegemma does NOT inject wheels (install_test_dependencies returns early); the sandbox venv reaches the host through a `_host_env.pth` listing every\n    host sys.path entry that contains 'site-packages'; the venv also has include-system-site-packages, and entries added by that .pth come BEFORE the system dist-packages,\n    so anything in the side directory OVERRIDES the host's copy;\n  * therefore: never put pytest, its core dependencies or pytest PLUGINS in the side directory (a plugin such as pytest-sanic is auto-loaded for every task and crashes\n    importing a package that is not installed). The harness itself skips pytest, pytest_timeout, pytest_asyncio, pytest_httpbin, pytest_xdist, pluggy, pip, setuptools, wheel.\n\"\"\"\n\nimport re\nimport subprocess\nimport sys\nfrom pathlib import Path\n\nDENY = {\"pytest\", \"pytest_timeout\", \"pytest_asyncio\", \"pytest_httpbin\", \"pytest_xdist\", \"pluggy\", \"pip\", \"setuptools\", \"wheel\", \"iniconfig\", \"packaging\", \"exceptiongroup\", \"tomli\"}\nPLUGIN_WHITELIST = {\"pytest_mock\", \"pytest_cov\", \"pytest_asyncio\", \"pytest_xdist\"}   # entry-point plugins that are inert unless the tests ask for them\n\n\ndef norm(name: str) -> str:\n    return re.sub(r\"[-_.]+\", \"_\", name).lower()\n\n\ndef classify(wheel: Path) -> str:\n    \"\"\"'ok' | 'deny' | 'plugin'. A pytest PLUGIN package (name starts with pytest_ / pytest-) is excluded unless whitelisted: its entry point is auto-loaded for every task.\n    Ordinary libraries that merely ship a pytest11 entry point (platformdirs, hypothesis, ...) are fine: they are installed anyway, so their plugin imports cleanly.\"\"\"\n    pkg = norm(wheel.name.split(\"-\", 1)[0])\n    if pkg in DENY:\n        return \"deny\"\n    if pkg.startswith(\"pytest_\") and pkg not in PLUGIN_WHITELIST:\n        return \"plugin\"\n    return \"ok\"\n\n\ndef build_taskdeps(wheels_dir: Path, target: Path) -> dict:\n    wheels = sorted(Path(wheels_dir).glob(\"*.whl\"))\n    kinds = {w: classify(w) for w in wheels}\n    use = [str(w) for w, k in kinds.items() if k == \"ok\"]\n    target.mkdir(parents=True, exist_ok=True)\n    r = subprocess.run([sys.executable, \"-m\", \"pip\", \"install\", \"-q\", \"--no-index\", \"--no-deps\", \"--target\", str(target), *use], capture_output=True, text=True)\n    return {\"rc\": r.returncode, \"installed\": len(use), \"denied\": sorted(w.name for w, k in kinds.items() if k == \"deny\"),\n            \"plugins_excluded\": sorted(w.name for w, k in kinds.items() if k == \"plugin\"), \"stderr_tail\": (r.stderr or \"\")[-300:]}\n")   # defines build_taskdeps / classify (pytest, its core deps and pytest plugins must NOT go into the side dir; see scripts/g9/taskdeps.py)
TD = Path('/tmp/taskdeps/site-packages')
_rep = build_taskdeps(POOL / 'wheels', TD)
print('taskdeps: rc', _rep['rc'], '| installed', _rep['installed'], '| denied', len(_rep['denied']), _rep['denied'][:12], '| pytest plugins excluded', len(_rep['plugins_excluded']), _rep['plugins_excluded'][:12])
assert _rep['rc'] == 0, _rep['stderr_tail']
sys.path.append(str(TD))
reg = ModelRegistry()
for m in discover_declared_models(ARM): reg.register(m, LiteLlm(model='openai/' + m, api_base='http://127.0.0.1:8000/v1', api_key='EMPTY'))
config = EvalConfig(tasks_path=POOL / 'tasks.jsonl', snapshots_dir=POOL / 'snapshots', results_dir=WORK / 'gold', submission_dir=ARM, models=reg, sandbox='subprocess',
    graph_dir=str(WORK / 'no_graphs'), embeddings_dir=str(WORK / 'no_embeddings'), wheels_dir=POOL / 'wheels')
evaluator = Evaluator(config)
tasks = load_tasks(config.tasks_path)
LIMIT = 12
if LIMIT: tasks = tasks[:LIMIT]
print('tasks:', len(tasks), flush=True)
results = {}
sem = asyncio.Semaphore(4)
BUDGET_S = 1.5 * 3600; T0 = time.time()

async def one(task, patch):
    t0 = time.perf_counter()
    try:
        snap, base, patch_path = resolve_task_snapshot_paths(config.snapshots_dir, task.instance_id, task.repo)
        res = await verify_task(evaluator.docker, config, task, snap, base_snapshot_path=base, patch_path=patch_path, agent_patch=patch, agent_error=None,
                                trace=SessionTrace(), start_time=t0)
        return {'resolved': bool(res.resolved), 'error': res.error, 'exit': res.test_exit_code, 'tail': (res.test_output or '')[-1200:] if not res.resolved else '', 'sec': round(time.perf_counter() - t0, 1)}
    except Exception as e:
        return {'resolved': False, 'error': f'{type(e).__name__}: {e}'[:400], 'exit': -1, 'tail': '', 'sec': round(time.perf_counter() - t0, 1)}

async def check(task):
    async with sem:
        if time.time() - T0 > BUDGET_S:
            results[task.instance_id] = {'repo': task.repo, 'gold': {'resolved': False, 'error': 'skipped_budget'}}; return
        g = await one(task, task.patch)
        rec = {'repo': task.repo, 'gold': g}
        if g['resolved']:
            rec['empty'] = await one(task, '')
        results[task.instance_id] = rec
        done = len(results)
        print(f"[{done}/{len(tasks)}] {task.instance_id}: gold {'PASS' if g['resolved'] else 'FAIL'} ({g['sec']}s)" + (f" | empty {'RESOLVED(!)' if rec['empty']['resolved'] else 'fails (good)'}" if 'empty' in rec else ''), flush=True)
        (WORK / 'gold_results.json').write_text(json.dumps(results, indent=1), encoding='utf-8')

async def main():
    await asyncio.gather(*(check(t) for t in tasks))

try:
    loop = asyncio.get_running_loop()
except RuntimeError:
    loop = None
if loop is not None and loop.is_running():
    import concurrent.futures
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
        pool.submit(lambda: asyncio.run(main())).result()
else:
    asyncio.run(main())

(WORK / 'gold_results.json').write_text(json.dumps(results, indent=1), encoding='utf-8')
good = sorted(i for i, r in results.items() if r['gold']['resolved'] and 'empty' in r and not r['empty']['resolved'])
(WORK / 'rollout_pool.json').write_text(json.dumps(good), encoding='utf-8')
print('\n===== harness gold check on the new pool =====')
print(f"gold passes {sum(r['gold']['resolved'] for r in results.values())}/{len(results)} | empty patch also resolved {sum(1 for r in results.values() if 'empty' in r and r['empty']['resolved'])} | ROLLOUT POOL {len(good)} tasks, {len({results[i]['repo'] for i in good})} repos")
errs = {}
for i, r in results.items():
    if not r['gold']['resolved']: errs.setdefault((r['gold']['error'] or 'tests failed')[:90], []).append(i)
print('\nFAIL reasons:')
for k, v in sorted(errs.items(), key=lambda kv: -len(kv[1]))[:20]: print(f'  {len(v):3d}  {k}   e.g. {v[:2]}')